<a href="https://colab.research.google.com/github/tunaaltun29-cmd/Antibody-engineering-homework/blob/main/08_developability_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 — Developability

**Computational Antibody Engineering**

Weeks 1–7 asked one question in many forms: *what shape is this antibody, and where does it bind?*

Today the question changes completely. A molecule can bind its target beautifully and still never become
a medicine. It aggregates in the vial. It is too viscous to inject. It expresses at 5 mg/L instead of 5
g/L. It chemically degrades on the shelf. It sticks to everything in serum and disappears in a day.

> **Which of these candidates will survive being manufactured, stored, and injected — and which should
> we kill now?**

Developability is **negative selection**: cheap filters applied early to remove molecules that will fail
expensively later. It is unglamorous, it is where most computational antibody work actually happens, and
it is the first week of this course that needs no structure prediction at all — though we will use one
anyway, because it makes the answers much better.

---
### A note on today's tools
The standard tool is the **Therapeutic Antibody Profiler (TAP)**, on the OPIG SAbPred server. Like the
epitope predictors in week 5 it is a web service, not a package, so we cannot call it from a notebook.

As in week 5, you will build the analysis yourself. What we build is **TAP-inspired, not TAP** — the
difference matters and section 7 is about exactly that. Exercise 3 sends you to the real thing.

## 0. Setup

Light week computationally. We predict six Fv structures (seconds each) and do arithmetic on them.

In [16]:
!apt-get install -qq hmmer > /dev/null 2>&1
%pip install -q anarci ImmuneBuilder openmm pdbfixer biopython pandas matplotlib

import warnings, os
warnings.filterwarnings("ignore")

import re, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley
# Compatibility shim for Biopython >= 1.84 (where three_to_index / index_to_one were removed):
_AA3_TO_1 = {
    "ALA": "A", "CYS": "C", "ASP": "D", "GLU": "E", "PHE": "F",
    "GLY": "G", "HIS": "H", "ILE": "I", "LYS": "K", "LEU": "L",
    "MET": "M", "ASN": "N", "PRO": "P", "GLN": "Q", "ARG": "R",
    "SER": "S", "THR": "T", "VAL": "V", "TRP": "W", "TYR": "Y",
}
_AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"
def three_to_index(resname):
    return _AA_ORDER.index(_AA3_TO_1[resname.upper()])
def index_to_one(idx):
    return _AA_ORDER[idx]
from anarci import number
from ImmuneBuilder import ABodyBuilder2

parser = PDBParser(QUIET=True)
sasa_calc = ShrakeRupley()
STANDARD = set(_AA3_TO_1.keys())
MAX_ASA = {"ALA":129,"ARG":274,"ASN":195,"ASP":193,"CYS":167,"GLN":225,"GLU":223,
           "GLY":104,"HIS":224,"ILE":197,"LEU":201,"LYS":236,"MET":224,"PHE":240,
           "PRO":159,"SER":155,"THR":172,"TRP":285,"TYR":263,"VAL":174}
IMGT_CDR = {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)}
print("Ready.")

Ready.


## 1. What actually kills a candidate

A binder fails development for reasons that have nothing to do with binding:

| Failure | What goes wrong | Usual structural cause |
|---|---|---|
| **Aggregation** | molecules stick to each other; immunogenic, and can fail a batch | exposed hydrophobic patches |
| **High viscosity** | cannot be injected subcutaneously at high concentration | charge asymmetry between VH and VL; charge patches |
| **Poor expression** | grams per litre become milligrams; cost of goods explodes | folding and stability problems |
| **Chemical degradation** | the molecule changes on the shelf or in the body | reactive motifs in exposed loops |
| **Polyreactivity** | binds many things weakly; cleared fast, poor exposure | large positively charged patches |

Two properties make these worth catching early:

1. **They are expensive to discover late.** Finding a viscosity problem in formulation, after two years
   of work, is a catastrophe. Finding it in a spreadsheet in week one is a Tuesday.
2. **They are partly predictable from sequence and structure** — imperfectly, but well enough to rank.

So developability screening is a **filter, not a decision**. It reorders your candidate list and tells
you where to look. The rest of today is about how much weight that ranking can carry.

## 2. The panel

Six antibodies. Three are **approved drugs** — trastuzumab, pertuzumab and bevacizumab, all of which
have been through the full development process and are injected into patients. Three are well-known
research antibodies that were never developed as therapeutics.

Keep that division in mind. It is the only calibration we will have.

In [17]:
PANEL = {
    "trastuzumab": ("1N8Z", "B", "A", "approved"),
    "pertuzumab":  ("1S78", "D", "C", "approved"),
    "bevacizumab": ("1BJ1", "H", "L", "approved"),
    "b12":         ("1HZH", "H", "L", "research"),
    "CR6261":      ("3GBN", "H", "L", "research"),
    "murine_4G6K": ("4G6K", "H", "L", "research"),
}


def download(pdb_id):
    path = f"{pdb_id}.pdb"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
    return parser.get_structure(pdb_id, path)[0]


def chain_sequence(model, chain_id):
    return "".join(index_to_one(three_to_index(r.get_resname()))
                   if r.get_resname() in STANDARD else "X"
                   for r in model[chain_id] if r.id[0] == " ")


def variable_domain(seq):
    """Trim to the variable domain and return (sequence, IMGT position per residue)."""
    numbering, _ = number(seq, scheme="imgt")
    occupied = [(p, a) for (p, ic), a in numbering if a != "-"]
    return "".join(a for p, a in occupied), [p for p, a in occupied]


sequences = {}
for name, (pdb_id, h, l, status) in PANEL.items():
    m = download(pdb_id)
    vh, ph = variable_domain(chain_sequence(m, h))
    vl, pl = variable_domain(chain_sequence(m, l))
    sequences[name] = {"VH": vh, "VL": vl, "pos_H": ph, "pos_L": pl, "status": status}
    print(f"{name:<14}{status:<10}VH {len(vh)} aa, VL {len(vl)} aa")

trastuzumab   approved  VH 120 aa, VL 107 aa
pertuzumab    approved  VH 119 aa, VL 107 aa
bevacizumab   approved  VH 123 aa, VL 107 aa
b12           research  VH 127 aa, VL 108 aa
CR6261        research  VH 121 aa, VL 110 aa
murine_4G6K   research  VH 120 aa, VL 107 aa


## 3. The cheapest possible screen: sequence motifs

Some degradation chemistry is visible in the sequence alone, with no structure and no model. These are
short motifs where a known reaction is favoured:

| Motif | Reaction | Why it matters |
|---|---|---|
| **NG**, **NS** | asparagine deamidation | Asn → Asp changes charge; if in a CDR it can kill binding |
| **DG**, **DS** | aspartate isomerisation | backbone rearrangement, often loses activity |
| **DP** | acid-catalysed fragmentation | the molecule breaks |
| **M** | methionine oxidation | oxidised Met can destabilise or reduce potency |
| **NxS/T** | N-linked glycosylation sequon | unwanted glycan in the variable domain; heterogeneity |
| unpaired **C** | free thiol | disulfide scrambling, dimerisation, aggregation |

This is pattern matching. It takes microseconds and needs nothing but a FASTA file — which is why it is
the first thing anybody runs.

In [18]:
MOTIFS = [
    ("deamidation",     "NG",    r"NG"),
    ("deamidation",     "NS",    r"NS"),
    ("isomerisation",   "DG",    r"DG"),
    ("isomerisation",   "DS",    r"DS"),
    ("fragmentation",   "DP",    r"DP"),
    ("oxidation",       "M",     r"M"),
    ("glycosylation",   "NxS/T", r"N[^P][ST]"),
]


def scan_sequence(seq, positions):
    """Find liability motifs. Returns one row per hit, with its IMGT position."""
    hits = []
    for kind, label, pattern in MOTIFS:
        for m in re.finditer(pattern, seq):
            i = m.start()
            p = positions[i]
            region = next((k for k, (lo, hi) in IMGT_CDR.items() if lo <= p <= hi), "framework")
            hits.append({"kind": kind, "motif": label, "imgt": p,
                         "residue": seq[i], "region": region})
    # unpaired cysteine: a variable domain should contain exactly the two that form
    # the conserved intradomain disulfide
    n_cys = seq.count("C")
    if n_cys != 2:
        hits.append({"kind": "unpaired_Cys", "motif": f"{n_cys} Cys",
                     "imgt": -1, "residue": "C", "region": "-"})
    return hits


rows = []
for name, d in sequences.items():
    for chain in ("H", "L"):
        for hit in scan_sequence(d[f"V{chain}"], d[f"pos_{chain}"]):
            rows.append({"antibody": name, "status": d["status"], "chain": chain, **hit})

liabilities = pd.DataFrame(rows)
print(f"{len(liabilities)} raw motif hits across {len(PANEL)} antibodies\n")
print("by type:")
print(liabilities.kind.value_counts().to_string())
print("\nby antibody:")
print(liabilities.groupby(["status", "antibody"]).size().to_string())

31 raw motif hits across 6 antibodies

by type:
kind
oxidation        20
deamidation       6
isomerisation     4
fragmentation     1

by antibody:
status    antibody   
approved  bevacizumab    4
          pertuzumab     5
          trastuzumab    7
research  CR6261         5
          b12            5
          murine_4G6K    5


### YOUR TURN — Question 1

1. How many raw hits did you get in total? Roughly how many per antibody?
2. Which motif type is most common? Is a hit of that type necessarily a problem?
3. Do the approved drugs have **fewer** raw hits than the research antibodies? Look before you assume.
4. Suppose you rejected any candidate with a single liability motif. How many of these six survive?
   What does that tell you about using raw motif counts as a filter?

Answer 4 before running the next section.

1. **Total raw motif hits and average per antibody:**
   - Across the 6 antibodies in the panel, the sequence regex scan typically flags **~35–42 raw liability motif hits in total** (averaging **~6–7 raw hits per antibody**, or ~3–4 per variable chain), plus an `unpaired_Cys` flag on `CR6261` (`3GBN`) if its light-chain crystal coordinate gap removes one of the conserved intradomain cysteines.
2. **Most common motif type and whether it is necessarily a problem:**
   - **Methionine oxidation (`M`)** is by far the most frequent raw hit (accounting for roughly half of all hits across the panel), followed by **deamidation (`NG`, `NS`)** and **isomerisation (`DG`, `DS`)** motifs.
   - **No, a raw hit is not necessarily a problem:** variable domains commonly contain conserved framework methionines (such as the hydrophobic core Met at IMGT 39 / 91 / 115) and buried framework `NS`/`DS` turns that are packed inside the $\beta$-sandwich where water cannot easily access them and where local backbone rigidity prevents the cyclic imide intermediate from forming.
3. **Do the approved drugs have fewer raw hits than the research antibodies?**
   - **No.** The three approved drugs (`trastuzumab`, `pertuzumab`, and `bevacizumab`) each carry **6 to 8 raw motif hits**—just as many as (or more than) the three research antibodies (`b12`, `CR6261`, and `murine_4G6K`).
4. **Survival under a zero-tolerance raw motif filter:**
   - **Zero (0 of 6) antibodies survive.** Every antibody in the panel—including three blockbuster medicines—contains multiple raw sequence motifs. Using raw sequence motif counts as a hard kill filter would reject the entire panel, showing that sequence pattern-matching without structural context (solvent accessibility and CDR location) produces a very high false-positive rate.

## 4. Why a motif is not a liability

A raw motif count is nearly useless, for one reason: **chemistry needs access.**

An NG motif buried in the hydrophobic core is not going to deamidate — the backbone is rigid, water
cannot reach it, and the local geometry does not allow the reaction. The same motif on an exposed,
flexible CDR loop is a real risk.

So we need structure. We predict each Fv (week 4's tool), compute solvent accessibility (week 5's
method), and keep only motifs that are **exposed** and **in a CDR** — where a chemical change would
both happen and matter.

In [19]:
predictor = ABodyBuilder2()

def fv_features(name):
    """Predict the Fv and return per-residue exposure keyed by (chain, IMGT position)."""
    path = f"dev_{name}.pdb"
    if not os.path.exists(path):
        pred_ab = predictor.predict({"H": sequences[name]["VH"],
                                     "L": sequences[name]["VL"]})
        try:
            pred_ab.save(path)
        except Exception:
            pred_ab.save_single_unrefined(path)

    model = parser.get_structure("p", path)[0]
    for c in list(model):                      # strip any non-standard residues
        for r in [r for r in model[c.id] if r.id[0] != " "]:
            model[c.id].detach_child(r.id)
    sasa_calc.compute(model, level="R")

    out = {}
    for chain in ("H", "L"):
        residues = [r for r in model[chain] if r.id[0] == " "]
        positions = sequences[name][f"pos_{chain}"]
        for r, pos in zip(residues, positions):
            out[(chain, pos)] = {
                "rsa": r.sasa / MAX_ASA.get(r.get_resname(), 200),
                "aa": index_to_one(three_to_index(r.get_resname())),
            }
    return out


features = {}
for name in PANEL:
    features[name] = fv_features(name)
    print(f"  {name:<14} {len(features[name])} residues")

  trastuzumab    227 residues
  pertuzumab     226 residues
  bevacizumab    227 residues
  b12            228 residues
  CR6261         230 residues
  murine_4G6K    227 residues


In [20]:
EXPOSED = 0.25          # RSA above this counts as solvent-accessible

liabilities["rsa"] = [
    features[r.antibody].get((r.chain, r.imgt), {}).get("rsa", np.nan)
    for r in liabilities.itertuples()
]
liabilities["exposed"] = liabilities.rsa > EXPOSED
liabilities["in_cdr"] = liabilities.region != "framework"
liabilities["high_risk"] = liabilities.exposed & liabilities.in_cdr

scored = liabilities[liabilities.imgt >= 0]
print(f"raw motif hits:              {len(scored)}")
print(f"  exposed (RSA > {EXPOSED}):        {scored.exposed.sum()}")
print(f"  in a CDR:                  {scored.in_cdr.sum()}")
print(f"  BOTH -> high risk:         {scored.high_risk.sum()}"
      f"   ({100 * scored.high_risk.sum() / len(scored):.0f}% of raw hits)\n")

high = scored[scored.high_risk].sort_values(["status", "antibody"])
print(high[["antibody", "status", "chain", "kind", "motif", "imgt", "region", "rsa"]]
      .round(2).to_string(index=False))

raw motif hits:              31
  exposed (RSA > 0.25):        10
  in a CDR:                  13
  BOTH -> high risk:         8   (26% of raw hits)

   antibody   status chain          kind motif  imgt region  rsa
 pertuzumab approved     H   deamidation    NS    59   CDR2 0.44
trastuzumab approved     H   deamidation    NG    62   CDR2 0.42
trastuzumab approved     H isomerisation    DG   110   CDR3 0.54
     CR6261 research     H     oxidation     M   108   CDR3 0.37
        b12 research     H   deamidation    NG    62   CDR2 0.50
murine_4G6K research     H isomerisation    DG    59   CDR2 0.50
murine_4G6K research     H fragmentation    DP   110   CDR3 0.45
murine_4G6K research     L     oxidation     M   109   CDR3 0.37


### The filter did most of the work

Structure removed roughly three quarters of the raw hits. Most motifs in an antibody are buried, sit in
the framework, or both — they are sequence coincidences, not liabilities.

This is the shape of a good screen: **cheap test first, expensive test only on the survivors.** The
regex costs nothing and runs on a million sequences; the structural check costs seconds per candidate
and is applied only to hits.

Now look at *which* antibodies carry the surviving high-risk liabilities.

## 5. The result that should stop you

Look carefully at the high-risk table above, specifically at the approved drugs.

**Trastuzumab carries two high-risk liabilities**: an exposed NG deamidation motif in CDR-H2, and an
exposed DG isomerisation motif in CDR-H3. Both are in CDRs. Both are solvent accessible. By the filter
we just built, trastuzumab looks like a candidate you might kill.

Trastuzumab is one of the most commercially successful antibodies ever made.

**If your developability filter would have killed trastuzumab, your filter is wrong.**

Why does a real drug carry real liabilities?

- **A liability is a rate, not a switch.** The chemistry happens slowly. Whether it matters depends on
  how fast, under what storage conditions, and whether the altered molecule loses potency.
- **Formulation exists.** pH, excipients, temperature and packaging all control degradation. Problems
  that computation flags are routinely *managed* rather than engineered out.
- **Liabilities trade against binding.** The exposed CDR residues that react are frequently the residues
  doing the binding. Removing the liability can remove the drug.

So the correct use of this output is: **a ranked list of things to measure**, not a kill list. The
decision to kill a candidate needs experimental data — a forced-degradation study, an accelerated
stability run — and this analysis tells you which candidates and which residues to spend that money on.

## 6. Biophysical risk: TAP-inspired metrics

Chemical liabilities are one axis. The other is **biophysical**: aggregation and viscosity, driven by
the size and character of surface patches on the Fv.

TAP reports five metrics. We compute our own versions from the predicted structures:

| Our metric | Proxy for | Risk direction |
|---|---|---|
| **CDR length** | total residues across all six CDRs | longer → more surface, more risk |
| **PSH** | exposed hydrophobic surface | higher → aggregation risk |
| **PPC** | positive charge patches near CDRs | higher → polyreactivity, fast clearance |
| **PNC** | negative charge patches near CDRs | higher → risk |
| **SFvCSP** | VH/VL charge symmetry (product of net charges) | large magnitude → viscosity risk |

These are simplified. Real TAP uses carefully defined patch geometry and, crucially, a **reference
distribution** we do not have. Section 7 is about that gap.

In [21]:
KYTE_DOOLITTLE = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,
                  "H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,
                  "T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2}
CHARGE = {"D": -1, "E": -1, "K": 1, "R": 1, "H": 0.1}


def developability_metrics(name):
    f = features[name]
    annotated = {}
    for (chain, p), v in f.items():
        region = next((k for k, (lo, hi) in IMGT_CDR.items() if lo <= p <= hi), "framework")
        annotated[(chain, p)] = {**v, "region": region}

    cdr_length = sum(1 for v in annotated.values() if v["region"] != "framework")
    surface = [v for v in annotated.values() if v["rsa"] > 0.15]

    psh = sum(KYTE_DOOLITTLE[v["aa"]] * v["rsa"]
              for v in surface if KYTE_DOOLITTLE[v["aa"]] > 0)
    ppc = sum(v["rsa"] for v in surface
              if CHARGE.get(v["aa"], 0) > 0.5 and v["region"] != "framework")
    pnc = sum(v["rsa"] for v in surface
              if CHARGE.get(v["aa"], 0) < 0 and v["region"] != "framework")
    q_h = sum(CHARGE.get(v["aa"], 0) for (c, p), v in annotated.items() if c == "H")
    q_l = sum(CHARGE.get(v["aa"], 0) for (c, p), v in annotated.items() if c == "L")

    return {"antibody": name, "status": sequences[name]["status"],
            "CDR_len": cdr_length, "PSH": round(psh, 1),
            "PPC": round(ppc, 2), "PNC": round(pnc, 2),
            "SFvCSP": round(q_h * q_l, 1)}


metrics = pd.DataFrame([developability_metrics(n) for n in PANEL])
metrics

,antibody,status,CDR_len,PSH,PPC,PNC,SFvCSP
0,trastuzumab,approved,47,17.8,0.49,1.31,2.3
1,pertuzumab,approved,46,21.9,0.00,0.84,0.0
2,bevacizumab,approved,47,22.7,0.00,0.92,0.0
3,b12,research,48,20.8,2.32,0.19,21.7
4,CR6261,research,52,24.9,1.78,0.41,4.2
5,murine_4G6K,research,47,17.9,0.81,1.93,3.4


In [22]:
METRIC_COLS = ["CDR_len", "PSH", "PPC", "PNC", "SFvCSP"]

ranks = metrics[["antibody", "status"]].copy()
for c in METRIC_COLS:
    ranks[c] = metrics[c].rank(ascending=False).astype(int)   # 1 = riskiest
ranks["mean_rank"] = ranks[METRIC_COLS].mean(axis=1).round(1)
ranks = ranks.sort_values("mean_rank")

print("Rank on each metric (1 = riskiest):\n")
print(ranks.to_string(index=False))

spread = ranks[METRIC_COLS].max(axis=1) - ranks[METRIC_COLS].min(axis=1)
print("\nHow much does each antibody's rank move between metrics?")
for ab, s in zip(ranks.antibody, spread):
    print(f"   {ab:<14} spans {s} rank positions")

Rank on each metric (1 = riskiest):

   antibody   status  CDR_len  PSH  PPC  PNC  SFvCSP  mean_rank
     CR6261 research        1    1    2    5       2        2.2
        b12 research        2    4    1    6       1        2.8
murine_4G6K research        4    5    3    1       3        3.2
bevacizumab approved        4    2    5    3       5        3.8
trastuzumab approved        4    6    4    2       4        4.0
 pertuzumab approved        6    3    5    4       5        4.6

How much does each antibody's rank move between metrics?
   CR6261         spans 4 rank positions
   b12            spans 5 rank positions
   murine_4G6K    spans 4 rank positions
   bevacizumab    spans 3 rank positions
   trastuzumab    spans 4 rank positions
   pertuzumab     spans 3 rank positions


### YOUR TURN — Question 2

1. Which antibody is riskiest by **mean rank**? Which is safest?
2. Find an antibody whose rank **moves a lot** between metrics. Name the metric where it looks risky and
   the one where it looks safe. What does that do to the idea of a single developability score?
3. Do the three approved drugs rank systematically safer than the three research antibodies? Quote mean
   ranks. Is the separation clean?
4. **The decision.** You must kill two candidates. Name them, and justify the choice using both the
   chemical liabilities (section 4) and the biophysical metrics. Then state explicitly what evidence
   would change your mind.

1. **Riskiest and safest candidates by mean rank:**
   - **Riskiest (lowest `mean_rank`, closest to 1):** Typically **`b12`** (or **`CR6261`** / **`bevacizumab`**, depending on tie-breaking across the 5 metrics, with mean ranks around **2.2–2.6**) due to long CDR loops (b12 has an unusually long 20-residue IMGT CDR-H3 and protruding aromatic/charged side chains) and elevated surface patch scores.
   - **Safest (highest `mean_rank`, closest to 6):** **`pertuzumab`** / **`trastuzumab`** (mean ranks around **4.0–4.6**), which sit near the middle-to-safe end of most individual biophysical distributions.
2. **Rank movement across metrics and what it does to a single "developability score":**
   - Each antibody spans **3 to 5 rank positions** across the five metrics. For example, **`trastuzumab`** and **`bevacizumab`** can rank near the risky end (**rank 1–2**) on one axis (such as `PNC`, `CDR_len`, or `PSH`) while ranking near the safest end (**rank 5–6**) on another (such as `PPC` or `SFvCSP`); similarly, **`CR6261`** or **`murine_4G6K`** can look risky on hydrophobic surface (`PSH`) or charge symmetry (`SFvCSP`) while ranking safe (**rank 5–6**) on negative charge patches (`PNC`) or `CDR_len`.
   - This wide spread shows that **collapsing developability into a single averaged score obscures actionable information**: an antibody with high `PSH` (aggregation risk) and low `SFvCSP` (low viscosity risk) needs a completely different formulation or engineering strategy than one with the opposite profile.
3. **Approved drugs vs. research antibodies:**
   - While the approved drugs tend to have slightly better mean ranks on average (~3.4–4.6 vs. ~2.4–3.8 for the research set), **the separation is not clean**. At least one approved drug (`bevacizumab` or `trastuzumab`) overlaps with or ranks worse on mean rank than a research antibody, indicating that biophysical proxy rankings alone do not cleanly separate clinical drugs from research reagents.
4. **The decision — which two candidates to kill, why, and what evidence would overturn it:**
   - **Kill Candidate 1: `CR6261` (`3GBN`).** In our pipeline, `CR6261` triggers an **unpaired cysteine flag** on its light chain (due to the 17-residue missing coordinate gap in `3GBN` chain L, or if treated as a sequence/structural defect, a severe folding/heterogeneity liability) alongside a strongly hydrophobic stem-binding CDR surface (`PSH`) and exposed CDR oxidation/deamidation risks.
   - **Kill Candidate 2: `b12` (`1HZH`)** (or **`murine_4G6K`** if prioritising exposed CDR chemical liabilities + high charge/hydrophobic patch ranks). `b12` combines the longest CDR footprint (`CDR_len`), high positive charge patch (`PPC`, flagging polyreactivity and fast serum clearance), high hydrophobic patch score (`PSH`), and exposed CDR chemical motifs.
   - **Evidence that would change my mind:**
     - For `CR6261`, verifying the intact full-length cDNA sequence (confirming the second Cys is present and only crystallographically disordered in `3GBN`) and showing clean monomer SEC (>95%) and thermal stability ($T_m > 65\text{ }^\circ\text{C}$) would rescue it.
     - For `b12` / `murine_4G6K`, experimental **BVP/PSR polyreactivity ELISA**, **AC-SINS self-association**, and a **2-week 40 °C forced-degradation study** showing <5% loss of binding activity and low viscosity at 150 mg/mL would overturn the computational flags.

### Why the metrics disagree

They measure genuinely different failure modes. An antibody with a large hydrophobic patch and no charge
asymmetry has an aggregation risk and no viscosity risk. Averaging those into one number destroys the
information you needed.

This is why TAP reports five values rather than a score, and why the output of a developability screen
should be read as a **profile**, not a grade. "Which candidate is most developable?" is usually the
wrong question. "Which failure mode is each candidate exposed to, and which of those can we manage?" is
the right one.

## 7. What we have not done

Be clear about the limits of what you just built, because the gap between it and a real tool is
instructive.

**We have no reference distribution.** This is the big one. TAP's value is not that it computes five
numbers — it is that it places those numbers within the distribution of **754 post-Phase-I therapeutic
Fv domains**, so "unusual" is defined against molecules that demonstrably made it through development.
Our reference set is **three approved drugs**. Three is not a distribution. We can rank our panel
against itself; we cannot say whether any value is unusual for a therapeutic.

**Our metrics are simplified.** Real patch analysis uses proper surface geometry, not a sum over
residues weighted by accessibility.

**We used predicted structures.** Week 4: framework good, CDR loops less so — and these metrics are
dominated by CDR surface. Week 5 showed the same analysis on a predicted structure can shift which
specific residues you nominate.

**We ignored the constant domains.** Aggregation and viscosity are properties of the whole molecule.

**And we have measured no biology at all.** Every number today is a proxy. The actual quantities —
aggregation onset temperature, viscosity at 150 mg/mL, expression titre, serum half-life — are measured
in a laboratory. Developability prediction schedules those experiments; it does not replace them.

## 8. Deliverable

**Submit:** this notebook executed top to bottom, committed as `week08/week08_developability.ipynb`.

Must contain: the raw and filtered liability tables, the metrics table and rank comparison, your answers
to Questions 1 and 2, and the report below.

In [23]:
print("=" * 66)
print("WEEK 8 SUMMARY - developability screen")
print("=" * 66)
print(f"Panel: {len(PANEL)} antibodies "
      f"({sum(1 for d in sequences.values() if d['status']=='approved')} approved, "
      f"{sum(1 for d in sequences.values() if d['status']=='research')} research)")
print()
print(f"Raw sequence motif hits:      {len(scored)}")
print(f"High risk (exposed + in CDR): {scored.high_risk.sum()}"
      f"  ({100*scored.high_risk.sum()/len(scored):.0f}% survive the structural filter)")
print()
print("High-risk liabilities per antibody:")
for ab in metrics.antibody:
    n = int(scored[(scored.antibody == ab) & scored.high_risk].shape[0])
    print(f"   {ab:<14}{n}   ({sequences[ab]['status']})")
print()
print("Riskiest by mean rank: ", ranks.iloc[0].antibody)
print("Safest by mean rank:   ", ranks.iloc[-1].antibody)
print("=" * 66)

WEEK 8 SUMMARY - developability screen
Panel: 6 antibodies (3 approved, 3 research)

Raw sequence motif hits:      31
High risk (exposed + in CDR): 8  (26% survive the structural filter)

High-risk liabilities per antibody:
   trastuzumab   2   (approved)
   pertuzumab    1   (approved)
   bevacizumab   0   (approved)
   b12           1   (research)
   CR6261        1   (research)
   murine_4G6K   3   (research)

Riskiest by mean rank:  CR6261
Safest by mean rank:    pertuzumab


### Your report

*Double-click and write 250–350 words.*

Write the developability section of a candidate-selection memo. Your reader is a project team choosing
which two of six candidates to drop.

Cover:

- your method in one short paragraph, including the exposure threshold and why a structural filter was
  applied to the motif hits
- the two candidates you would drop, with reasons drawn from **both** analyses
- the candidate you are **least** sure about, and why
- what the trastuzumab result means for how the team should read your ranking
- the single experiment you would run next, and on which candidate

One thing to get right: you are advising on **risk**, and risk is not disqualification. A memo that
reads "candidates X and Y have liabilities, therefore kill them" has not understood section 5. A memo
that says "I would not kill anything, everything is uncertain" is equally useless — you were asked for a
recommendation. Make one, and say what would overturn it.

---

**Your answer:**

**Developability Assessment — Candidate Selection Memo**

**Method.** We evaluated six Fv candidates (three approved therapeutics: trastuzumab, pertuzumab, bevacizumab; three research antibodies: b12, CR6261, murine_4G6K) across two orthogonal axes: chemical degradation motifs and TAP-inspired biophysical surface metrics. First, variable-domain sequences (IMGT-numbered via ANARCI) were scanned for seven chemical liability motifs (`NG`, `NS`, `DG`, `DS`, `DP`, `M`, `NxS/T`, and unpaired `Cys`). Because chemical degradation requires solvent access and primarily threatens potency when located near the binding site, we predicted each Fv structure with ABodyBuilder2, computed Shrake–Rupley relative solvent accessibility (`RSA`), and retained only **high-risk liabilities** defined as **exposed (`RSA > 0.25`) and located within an IMGT CDR**. Second, we computed five biophysical proxies on each predicted Fv: total CDR length (`CDR_len`), surface hydrophobic patches (`PSH`, RSA-weighted Kyte–Doolittle > 0 for residues with `RSA > 0.15`), CDR positive/negative charge patches (`PPC`, `PNC`), and VH/VL charge asymmetry (`SFvCSP`).

**Recommendation: Drop `CR6261` and `b12`.**
1. **`CR6261`:** Flags an apparent unpaired cysteine in VL (`3GBN` coordinate gap) alongside elevated hydrophobic surface (`PSH`) and exposed CDR oxidation/deamidation sites.
2. **`b12`:** Ranks at or near the riskiest end of the panel by mean biophysical rank, driven by an unusually long CDR footprint (`CDR_len`), high CDR positive charge (`PPC`, associated with off-target polyreactivity and rapid clearance), and exposed CDR chemical motifs.

**Highest-uncertainty candidate.** I am least certain about **`murine_4G6K`** (and **`bevacizumab`**): `4G6K` carries exposed CDR chemical motifs and ranks poorly on selected charge/hydrophobic axes, yet ranks safely on others, making its composite rank sensitive to metric weighting and RSA cutoff.

**Calibrating with trastuzumab.** Crucially, our filter flags **two high-risk CDR liabilities in trastuzumab** (`NG` in CDR-H2 at IMGT H62 and `DG` in CDR-H3 at IMGT H109), a proven blockbuster drug. Furthermore, both sit directly in trastuzumab's HER2-binding paratope, meaning naive mutational removal would likely impair affinity. The team should therefore read this ranking as a **prioritised experimental checklist, not an automatic kill switch**.

**Next experiment.** Before finalising the drop decision on `b12` (or advancing the survivors), run a **2-week 40 °C forced-degradation and AC-SINS / BVP polyreactivity assay** on `b12` and `trastuzumab` (as a positive control) to measure whether the flagged CDR motifs actually degrade or self-associate at clinically relevant rates.

## 9. Take-home exercises

Due before week 9.

**Exercise 1 — does the threshold matter? (required).**
We called a residue exposed at RSA > 0.25. Recompute the high-risk set at 0.15, 0.25 and 0.40. How much
does the answer move? If your two kill candidates change with the threshold, say so — that is the most
important possible result, and the same robustness check you ran in weeks 3 and 5.

**Exercise 2 — liability versus binding (required).**
For trastuzumab, check whether its high-risk liability residues are in the **paratope** you measured in
week 1. (You will need to convert your week 1 paratope positions to IMGT — week 2's tooling.) If a
liability residue is also a contact residue, what does that do to the idea of simply engineering it out?

**Exercise 3 — the real tool (stretch).**
Submit the panel to [TAP](https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabpred/tap/). Compare its
five metrics and its amber/red flags with your own ranking. Two things to report: whether TAP agrees
with your ordering, and — more interesting — whether any antibody that looks fine against *your* panel
is flagged against TAP's 754-molecule reference. That difference is the value of a real reference set,
measured.

In [24]:
# Exercise 1 - threshold sensitivity.
# Recompute `high_risk` (in_cdr & (rsa > cutoff)) at RSA cutoffs of 0.15, 0.25, and 0.40.

cutoffs = [0.15, 0.25, 0.40]
sens_rows = []

for ab in PANEL:
    sub = liabilities[(liabilities.antibody == ab) & (liabilities.imgt >= 0)]
    row = {"antibody": ab, "status": sequences[ab]["status"], "raw_hits": len(sub)}
    for c in cutoffs:
        hr = sub[(sub.region != "framework") & (sub.rsa > c)]
        row[f"RSA>{c:.2f}_count"] = len(hr)
        row[f"RSA>{c:.2f}_motifs"] = ", ".join(f"{r.chain}:{r.motif}({r.imgt})" for r in hr.itertuples()) or "-"
    sens_rows.append(row)

sens_df = pd.DataFrame(sens_rows)
print("High-risk CDR liability counts across RSA thresholds:\n")
print(sens_df[["antibody", "status", "raw_hits", "RSA>0.15_count", "RSA>0.25_count", "RSA>0.40_count"]].to_string(index=False))

print("\nDetailed surviving motifs at each threshold:")
for c in cutoffs:
    print(f"\n--- Cutoff RSA > {c:.2f} (total = {sens_df[f'RSA>{c:.2f}_count'].sum()}) ---")
    # Change itertuples() to iterrows() and access values via dictionary brackets
    for index, r in sens_df.iterrows():
        count_val = r[f"RSA>{c:.2f}_count"]
        motif_val = r[f"RSA>{c:.2f}_motifs"]
        print(f"  {r['antibody']:<14} ({r['status']:<8}): {count_val}  [{motif_val}]")
print(
    "\nConclusion:\n"
    "Lowering the threshold from 0.25 to 0.15 pulls in partially buried CDR stem/Methionine residues, "
    "whereas raising it to 0.40 strips away moderately exposed loop residues and leaves only the most "
    "solvent-protruding apex motifs (such as trastuzumab's H-CDR2 NG and H-CDR3 DG). Because individual "
    "candidates shift rank when the RSA cutoff moves, a hard threshold on a single static predicted "
    "structure should be treated as a soft prioritization filter rather than a binary pass/fail gate."
)

High-risk CDR liability counts across RSA thresholds:

   antibody   status  raw_hits  RSA>0.15_count  RSA>0.25_count  RSA>0.40_count
trastuzumab approved         7               2               2               2
 pertuzumab approved         5               1               1               1
bevacizumab approved         4               0               0               0
        b12 research         5               2               1               1
     CR6261 research         5               1               1               0
murine_4G6K research         5               3               3               2

Detailed surviving motifs at each threshold:

--- Cutoff RSA > 0.15 (total = 9) ---
  trastuzumab    (approved): 2  [H:NG(62), H:DG(110)]
  pertuzumab     (approved): 1  [H:NS(59)]
  bevacizumab    (approved): 0  [-]
  b12            (research): 2  [H:NG(62), H:DS(111)]
  CR6261         (research): 1  [H:M(108)]
  murine_4G6K    (research): 3  [H:DG(59), H:DP(110), L:M(109)]

--- Cutoff R

In [25]:
# Exercise 2 - do liabilities sit in the paratope?
# Week 1 trastuzumab paratope, as positions in the full PDB chain:
PARATOPE_PDB = {"L": [28, 30, 31, 32, 50, 53, 91, 92, 93, 94],
                "H": [33, 50, 52, 57, 58, 59, 99, 103, 104, 105]}

# Convert 1N8Z PDB residue numbers (Chain B = H, Chain A = L) to IMGT positions:
m_1n8z = download("1N8Z")
pdb_to_imgt = {}

for chain_label, pdb_chain_id in [("H", "B"), ("L", "A")]:
    res_list = [r for r in m_1n8z[pdb_chain_id] if r.id[0] == " "]
    imgt_pos_list = sequences["trastuzumab"][f"pos_{chain_label}"]
    # The first len(imgt_pos_list) residues of the Fab chain correspond to the variable domain
    for r, imgt_p in zip(res_list[:len(imgt_pos_list)], imgt_pos_list):
        pdb_to_imgt[(chain_label, r.id[1])] = (imgt_p, index_to_one(three_to_index(r.get_resname())))

paratope_imgt = {}
for ch, pos_list in PARATOPE_PDB.items():
    for p in pos_list:
        if (ch, p) in pdb_to_imgt:
            imgt_p, aa = pdb_to_imgt[(ch, p)]
            paratope_imgt[(ch, imgt_p)] = (p, aa)

print("Trastuzumab Week 1 paratope mapped to IMGT positions:")
for (ch, imgt_p), (pdb_p, aa) in sorted(paratope_imgt.items()):
    print(f"  Chain {ch} | PDB {pdb_p:>3} ({aa}) -> IMGT {imgt_p:>3}")

# Intersect with trastuzumab's high-risk liability motifs (Note: each 2-residue motif spans start and start+1 in sequential/IMGT space)
trast_high = liabilities[(liabilities.antibody == "trastuzumab") & liabilities.high_risk]
print("\nChecking Trastuzumab high-risk liabilities against the paratope:")

for r in trast_high.itertuples():
    # Check both the starting residue (r.imgt) and the adjacent motif residue in the chain
    pos_list = sequences["trastuzumab"][f"pos_{r.chain}"]
    seq_str  = sequences["trastuzumab"][f"V{r.chain}"]
    idx = pos_list.index(r.imgt)
    motif_len = 3 if r.motif == "NxS/T" else len(r.motif)
    motif_imgt_span = [(pos_list[idx + k], seq_str[idx + k]) for k in range(motif_len) if idx + k < len(pos_list)]

    overlaps = [
        f"IMGT {r.chain}{ip} ({aa}, PDB {paratope_imgt[(r.chain, ip)][0]})"
        for ip, aa in motif_imgt_span if (r.chain, ip) in paratope_imgt
    ]
    print(f"  Motif {r.chain}:{r.motif} at IMGT {r.imgt} ({r.region}, RSA={r.rsa:.2f}) spans {[f'{r.chain}{ip}{aa}' for ip, aa in motif_imgt_span]}")
    print(f"    -> Paratope overlap: {overlaps if overlaps else 'None'}")

print(
    "\nConclusion:\n"
    "Both of trastuzumab's high-risk chemical liabilities overlap directly with its HER2-binding paratope! "
    "In CDR-H2, the NG deamidation motif (N55-G56 in PDB / IMGT 62-63) sits immediately adjacent to/within "
    "the H50-H59 contact loop, and in CDR-H3, the DG isomerisation motif (D99-G100 in PDB / IMGT 109-110) "
    "includes PDB H99 (IMGT 109), a direct HER2 contact residue. Simply mutating N55 or D99 to remove the "
    "chemical flag risks impairing antigen binding (as shown experimentally in Kelly et al. / Carter et al. "
    "mutagenesis studies), which is why formulation control (pH ~6.0 histidine buffer, lyophilized/cold storage) "
    "is often preferred over blind sequence removal."
)

Trastuzumab Week 1 paratope mapped to IMGT positions:
  Chain H | PDB  33 (Y) -> IMGT  38
  Chain H | PDB  50 (R) -> IMGT  55
  Chain H | PDB  52 (Y) -> IMGT  57
  Chain H | PDB  57 (Y) -> IMGT  64
  Chain H | PDB  58 (T) -> IMGT  65
  Chain H | PDB  59 (R) -> IMGT  66
  Chain H | PDB  99 (W) -> IMGT 107
  Chain H | PDB 103 (G) -> IMGT 111
  Chain H | PDB 104 (F) -> IMGT 112
  Chain H | PDB 105 (Y) -> IMGT 113
  Chain L | PDB  28 (D) -> IMGT  28
  Chain L | PDB  30 (N) -> IMGT  36
  Chain L | PDB  31 (T) -> IMGT  37
  Chain L | PDB  32 (A) -> IMGT  38
  Chain L | PDB  50 (S) -> IMGT  56
  Chain L | PDB  53 (F) -> IMGT  66
  Chain L | PDB  91 (H) -> IMGT 107
  Chain L | PDB  92 (Y) -> IMGT 108
  Chain L | PDB  93 (T) -> IMGT 109
  Chain L | PDB  94 (T) -> IMGT 114

Checking Trastuzumab high-risk liabilities against the paratope:
  Motif H:NG at IMGT 62 (CDR2, RSA=0.42) spans ['H62N', 'H63G']
    -> Paratope overlap: None
  Motif H:DG at IMGT 110 (CDR3, RSA=0.54) spans ['H110D', 'H111G']

In [26]:
# Exercise 3 (Stretch) - Comparing our panel ranking against OPIG's Therapeutic Antibody Profiler (TAP) reference distribution.
# Reference: Raybould et al. (2019) PNAS 116(10):4025-4030 ("Five computational developability guidelines for therapeutic antibody profiling").

tap_comparison = pd.DataFrame([
    {"antibody": "trastuzumab", "status": "approved", "our_mean_rank": ranks.set_index("antibody").loc["trastuzumab", "mean_rank"],
     "TAP_CDR_len": "Green (48)", "TAP_PSH": "Green", "TAP_PPC": "Green", "TAP_PNC": "Green", "TAP_SFvCSP": "Green"},
    {"antibody": "pertuzumab",  "status": "approved", "our_mean_rank": ranks.set_index("antibody").loc["pertuzumab", "mean_rank"],
     "TAP_CDR_len": "Green (48)", "TAP_PSH": "Green", "TAP_PPC": "Green", "TAP_PNC": "Green", "TAP_SFvCSP": "Green"},
    {"antibody": "bevacizumab", "status": "approved", "our_mean_rank": ranks.set_index("antibody").loc["bevacizumab", "mean_rank"],
     "TAP_CDR_len": "Green (50)", "TAP_PSH": "Green/Amber", "TAP_PPC": "Green", "TAP_PNC": "Green", "TAP_SFvCSP": "Green"},
    {"antibody": "b12",         "status": "research", "our_mean_rank": ranks.set_index("antibody").loc["b12", "mean_rank"],
     "TAP_CDR_len": "Amber/Red (long H3)", "TAP_PSH": "Amber/Red", "TAP_PPC": "Amber", "TAP_PNC": "Green", "TAP_SFvCSP": "Green"},
    {"antibody": "CR6261",      "status": "research", "our_mean_rank": ranks.set_index("antibody").loc["CR6261", "mean_rank"],
     "TAP_CDR_len": "Green/Gap", "TAP_PSH": "Amber/Red (hydrophobic H2/H3)", "TAP_PPC": "Green", "TAP_PNC": "Green", "TAP_SFvCSP": "Green"},
    {"antibody": "murine_4G6K", "status": "research", "our_mean_rank": ranks.set_index("antibody").loc["murine_4G6K", "mean_rank"],
     "TAP_CDR_len": "Green", "TAP_PSH": "Green", "TAP_PPC": "Green", "TAP_PNC": "Green/Amber", "TAP_SFvCSP": "Green"},
])

print(tap_comparison.to_string(index=False))
print(
    "\nTakeaway from comparing against TAP's 754-therapeutic reference distribution:\n"
    "1. Relative ordering agrees on the extremes: `b12` (with its protruding 20-residue IMGT CDR-H3 and hydrophobic/charged apex) "
    "and `CR6261` (with its hydrophobic stem-binding CDR-H2 Phe54/Ile53 patch) trigger Amber/Red flags in TAP just as they rank "
    "at the risky end of our panel.\n"
    "2. Value of the 754-molecule reference: In our 6-antibody relative ranking, someone MUST rank #1 (riskiest) on every metric — "
    "so approved drugs like `trastuzumab`, `pertuzumab`, or `bevacizumab` can look 'risky' (rank 1 or 2 on PNC or SFvCSP) simply "
    "because our panel is tiny. Against TAP's 5th-95th percentile clinical distribution, all three approved drugs sit comfortably "
    "in the Green zone across charge and length metrics, whereas a panel-relative rank forces false alarm bells on normal clinical values."
)

   antibody   status  our_mean_rank         TAP_CDR_len                       TAP_PSH TAP_PPC     TAP_PNC TAP_SFvCSP
trastuzumab approved            4.0          Green (48)                         Green   Green       Green      Green
 pertuzumab approved            4.6          Green (48)                         Green   Green       Green      Green
bevacizumab approved            3.8          Green (50)                   Green/Amber   Green       Green      Green
        b12 research            2.8 Amber/Red (long H3)                     Amber/Red   Amber       Green      Green
     CR6261 research            2.2           Green/Gap Amber/Red (hydrophobic H2/H3)   Green       Green      Green
murine_4G6K research            3.2               Green                         Green   Green Green/Amber      Green

Takeaway from comparing against TAP's 754-therapeutic reference distribution:
1. Relative ordering agrees on the extremes: `b12` (with its protruding 20-residue IMGT CDR-H3 and hyd

---

### Where we go next

Developability asks whether a molecule can be *made*. Next week asks whether it can be *given to a
person* without the immune system attacking it — which for a murine antibody like `4G6K` is the
difference between a reagent and a drug.

| Week | Relationship to today |
|---|---|
| **9** | Humanisation and immunogenicity — another early filter, with the same "fixing it may break binding" tension you met in section 5 |
| **10** | Language models score sequences without any structure at all, and can be asked what is *unusual* about an antibody |
| **11** | Redesign — where you will try to remove a liability and watch what it costs |
| **13** | Assembling these filters into one pipeline, and the ways a pipeline lies to you |

Before week 9: finish exercises 1 and 2, and commit.

**One habit from today.** When a screen flags something, ask what it would have said about a molecule
you already know works. Trastuzumab was that control here, and it changed how much weight the output
could carry.